# Chapter 10 — Clustering Techniques

**Nama:** Omar Al Farizy Hutagalung  

**NIM:** 101032300034  

**Mata Kuliah:** Machine Learning  

**Tugas:** Tugas 2 — Code Reproduction & Theoretical Deep-Dive

---

## Objective

Chapter ini membahas unsupervised learning untuk menemukan kelompok alami pada data, menggunakan K-Means, hierarchical clustering, dan DBSCAN serta metrik evaluasi clustering.

## 1. Landasan Teori

Clustering mengelompokkan sampel berdasarkan kemiripan fitur tanpa membutuhkan label kelas saat pelatihan.

- **K-Means:** membagi data menjadi sejumlah cluster yang ditentukan sebelumnya dengan meminimalkan variasi dalam cluster.
- **Hierarchical clustering:** membangun struktur cluster bertingkat.
- **DBSCAN:** mengidentifikasi cluster berdasarkan kepadatan dan dapat menandai titik noise.
- **Scaling:** penting ketika fitur memiliki skala berbeda karena jarak dapat didominasi fitur berskala besar.

Label cluster adalah identitas kelompok, bukan label kelas yang memiliki makna intrinsik.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score

X, y_reference = make_blobs(
    n_samples=400, centers=4, cluster_std=0.75, random_state=42
)
X_scaled = StandardScaler().fit_transform(X)

plt.scatter(X[:, 0], X[:, 1], s=20)
plt.title("Data sintetis sebelum clustering")
plt.xlabel("Fitur 1")
plt.ylabel("Fitur 2")
plt.show()

## 2. K-Means

Parameter `n_clusters` menentukan jumlah cluster. `random_state` membuat hasil lebih mudah direproduksi.

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

print("Ukuran cluster:")
print(pd.Series(kmeans_labels).value_counts().sort_index())
print("Silhouette score:", round(silhouette_score(X_scaled, kmeans_labels), 3))

plt.scatter(X[:, 0], X[:, 1], c=kmeans_labels, s=20)
plt.title("Hasil K-Means")
plt.xlabel("Fitur 1")
plt.ylabel("Fitur 2")
plt.show()

Silhouette score berada pada rentang -1 sampai 1. Nilai lebih tinggi umumnya menunjukkan cluster yang lebih kompak dan terpisah, tetapi interpretasi tetap bergantung pada bentuk dan karakter data.

## 3. Membandingkan algoritma clustering

In [ ]:
algorithms = {
    "K-Means": KMeans(n_clusters=4, random_state=42, n_init=10),
    "Agglomerative": AgglomerativeClustering(n_clusters=4),
    "DBSCAN": DBSCAN(eps=0.35, min_samples=5),
}

results = []
for name, algorithm in algorithms.items():
    labels = algorithm.fit_predict(X_scaled)
    cluster_ids = set(labels) - {-1}
    n_clusters = len(cluster_ids)
    n_noise = int(np.sum(labels == -1))
    valid = labels != -1

    score = np.nan
    if len(set(labels[valid])) >= 2 and valid.sum() > len(set(labels[valid])):
        score = silhouette_score(X_scaled[valid], labels[valid])

    results.append({
        "Algorithm": name,
        "Jumlah cluster (tanpa noise)": n_clusters,
        "Jumlah noise": n_noise,
        "Silhouette (tanpa noise)": score,
    })

display(pd.DataFrame(results).round(3))

## 4. Analisis dan Kesimpulan

- K-Means cocok untuk cluster yang relatif kompak dan memerlukan jumlah cluster yang ditentukan sebelumnya.
- Agglomerative clustering membentuk kelompok secara hierarkis.
- DBSCAN dapat mendeteksi noise, tetapi sensitif terhadap parameter `eps` dan `min_samples`.
- Silhouette score tidak selalu cocok untuk semua struktur cluster; visualisasi dan pengetahuan domain tetap diperlukan.

**Kesimpulan:** pemilihan algoritma clustering harus mempertimbangkan distribusi data, skala fitur, noise, serta tujuan analisis.